# FLYP on iWildCam: Kaggle reproduction

Reproduces the **iWildCam** experiment of *Finetune Like You Pretrain* (FLYP, `locuslab/FLYP`) with CLIP ViT-B/16. Defaults follow the paper/README setting (20 epochs, batch 256, lr 1e-5, wd 0.2), run with fp16 AMP so batch 256 fits 2x T4. The dataset comes from the team's HF mirror
[`emmilly/iwildcam-wilds-v2`](https://huggingface.co/datasets/emmilly/iwildcam-wilds-v2), an unmodified copy of the official WILDS v2.0 archive (12 GB).

**Kaggle settings before you run**
- Accelerator: **GPU** (T4 x2 or P100), **Internet: On** (needed for GitHub, HF Hub, CLIP weights).
- Needs ~35 GB of scratch disk (12 GB archive + ~12 GB extracted + checkpoints). Scratch lives in `/kaggle/temp`; it is not saved.
- Optional: add Kaggle secret `HF_TOKEN` (write access) and set `HF_LOG_REPO` below to also push logs to the Hub after every epoch.

**What is saved where (so results survive the run)**
- `/kaggle/working/results/<RUN_NAME>/` is kept as the notebook output: `notebook.log` (this notebook's progress), `train_stdout.log` (full training output), `expt_logs/**/stats.tsv` (per-epoch metrics, refreshed after every epoch), `env.json`, `config.json`, `repo_patches.diff`, `results_summary.json`.
- Checkpoints go to scratch (`/kaggle/temp`) and only the newest few are kept.

**Weights & Biases:** set Kaggle secret `WANDB_API_KEY` (Add-ons > Secrets; a plain env var of the same name also works) and set `WANDB_ENTITY` (team) and `WANDB_PROJECT` in the config cell. Step losses, per-epoch ID/OOD metrics and final per-split scores are logged per seed.

**WILDS leaderboard:** `PRESET = "leaderboard"` selects the leaderboard FLYP setting (CLIP ViT-L/14@336px, 20 epochs, batch 256, 3 seeds). After each seed the notebook predicts the `val/test/id_val/id_test` splits, re-scores them with the WILDS evaluator, and finally packs `wilds_submission.tar.gz` for the submission form. ViT-L/14@336px does **not** fit 2xT4 at batch 256 and one epoch would take about half a day, so this needs a large GPU (A100/H100 class); the baseline preset only produces a sanity-check submission file set.

**Flow:** config → install → clone FLYP and patch logging → download + verify → extract + check splits → build FLYP csv → smoke test (a few steps, catches OOM/path problems) → full training → summary.

In [1]:
# ============================== CONFIG (edit here) ==============================
import os, sys, json, time, shutil, re, subprocess, hashlib, tarfile, platform
from pathlib import Path
from datetime import datetime

RUN_NAME   = "iwildcam_flyp_vitb16"
EXP_NAME   = "iwildcam/flyp_loss"

# FLYP code, pinned to the commit the project was checked out at
REPO_URL    = "https://github.com/locuslab/FLYP.git"
REPO_COMMIT = "215d5bb6feeda6675f60e5818abcb4f6465c83af"

# Dataset mirror on the HF Hub
HF_DATASET_REPO = "emmilly/iwildcam-wilds-v2"
HF_ARCHIVE      = "iwildcam_v2.0.tar.gz"
ARCHIVE_SHA256  = "05c872f46eee44315f79bc90f4098c69ae3799a2b95116db02292ea1b4e370d2"
ARCHIVE_SIZE    = 11957415473

# Defaults follow the README iWildCam command (ViT-B/16, 20 epochs, batch 256, lr 1e-5, wd 0.2). Batch 256 only fits
# 2x16 GB GPUs with AMP. 20 epochs may not fit one 12 h Kaggle session: check the first epoch's ETA and, if needed, lower
# EPOCHS (the cosine schedule follows it) or run on bigger hardware. Earlier fp32 batch-64 run: ~1 h/epoch on 2x T4.
MODEL, EPOCHS, BATCH_SIZE, LR, WD = "ViT-B/16", 20, 64, 1e-5, 0.2   # = the paper/README iWildCam setting
AMP = True                             # fp16 autocast + GradScaler: needed for batch 256 on 2x16 GB GPUs, and faster on T4 tensor cores; the paper used fp32
SEEDS = [0]                            # one full training run per seed; the leaderboard needs >= 3

# "baseline": the cheap setting above.  "leaderboard": the WILDS leaderboard FLYP entry (CLIP ViT-L/14@336px;
# hyperparameters were tuned on ViT-B/16 and reused, see the leaderboard note). Needs a much bigger GPU than a T4.
PRESET = "baseline"
if PRESET == "leaderboard":
    MODEL, EPOCHS, BATCH_SIZE, LR, WD = "ViT-L/14@336px", 20, 256, 1e-5, 0.2
    SEEDS = [0, 1, 2]
DRY_RUN_STEPS = 0                      # >0: run the WHOLE pipeline (train, eval, predict, W&B, package) but only this many steps per epoch; set EPOCHS = 1 too
EXTRA_PRED_DIRS = []                   # prediction folders from earlier sessions (e.g. a previous notebook's output) to merge into the submission
WORKERS = os.cpu_count() or 4          # dataloader workers (Kaggle GPU boxes have 4 cores)

RUN_SMOKE_TEST_FIRST = True            # 3 train steps + 3 eval batches per split, same batch size
SMOKE_STEPS, SMOKE_EVAL_BATCHES = 3, 3
PRINT_EVERY = 25                       # training-step log interval (README code uses 100)
KEEP_LAST_CHECKPOINTS = 2              # epochs of checkpoints kept in scratch
SAVE_FINAL_CHECKPOINT_TO_OUTPUT = False  # copy last checkpoint (~0.6 GB) into /kaggle/working/results
# Weights & Biases. The API key is NOT set here: add Kaggle secret / env var WANDB_API_KEY.
WANDB_ENABLED = True
WANDB_ENTITY  = "team-support-vectors-idlf26"                     # W&B team (entity); "" = your default entity
WANDB_PROJECT = "iwilds"
HF_LOG_REPO = ""                     # e.g. "emmilly/flyp-iwildcam-logs" -> private dataset repo, needs secret HF_TOKEN

# ------------------------------- paths -------------------------------
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "work"
WORK.mkdir(parents=True, exist_ok=True)

def _pick_scratch():
    best = None
    for cand in ([Path("/kaggle/temp")] if ON_KAGGLE else []) + [Path("/tmp/flyp_scratch")]:
        try:
            cand.mkdir(parents=True, exist_ok=True)
            free = shutil.disk_usage(cand).free
        except OSError:
            continue
        if best is None or free > best[1]:
            best = (cand, free)
    return best

SCRATCH, _free = _pick_scratch()
RESULTS   = WORK / "results" / RUN_NAME
REPO      = WORK / "FLYP"
DATA_ROOT = SCRATCH / "data"
DATA_DIR  = DATA_ROOT / "iwildcam_v2.0"      # wilds expects <root>/iwildcam_v2.0
CKPT      = SCRATCH / "checkpoints"
RESULTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"scratch={SCRATCH} (free {_free/1e9:.0f} GB)  results={RESULTS}  repo={REPO}")

scratch=/kaggle/temp (free 1146 GB)  results=/kaggle/working/results/iwildcam_flyp_vitb16  repo=/kaggle/working/FLYP


In [2]:
# ============================ logging + subprocess helpers ============================
import pandas as pd

def log(msg):
    """Print and append to results/notebook.log (flushed immediately)."""
    line = f"{datetime.now():%Y-%m-%d %H:%M:%S} | {msg}"
    print(line, flush=True)
    with open(RESULTS / "notebook.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")

def fmt_t(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}:{sec % 3600 // 60:02d}:{sec % 60:02d}"

def run(cmd, log_path=None, cwd=None, env=None, hook=None, tail=40):
    """Run a command, stream every line to the notebook and to log_path. Raises with the tail on failure."""
    cmd = [str(c) for c in cmd]
    log("$ " + " ".join(cmd))
    p = subprocess.Popen(cmd, cwd=cwd, env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace")
    last = []
    f = open(log_path, "a", encoding="utf-8") if log_path else None
    try:
        for line in p.stdout:                       # universal newlines: tqdm's \r updates become lines
            line = line.rstrip("\n")
            if not line.strip():
                continue
            last = (last + [line])[-tail:]
            note = hook(line) if hook else None
            out = f"{line}   {note}" if note else line
            print(out, flush=True)
            if f:
                f.write(out + "\n"); f.flush()
    finally:
        if f:
            f.close()
    rc = p.wait()
    if rc != 0:
        hint = ""
        if any("out of memory" in l.lower() for l in last):
            hint = "\n>>> CUDA out of memory: lower BATCH_SIZE in the config cell (128, then 64) and re-run from there."
        raise RuntimeError(f"command failed with exit code {rc}. Last lines:\n" + "\n".join(last) + hint)

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

log(f"=== notebook start | run={RUN_NAME} | python {platform.python_version()} ===")

2026-10-21 12:49:09 | === notebook start | run=iwildcam_flyp_vitb16 | python 3.13.15 ===


In [3]:
# ============================== install dependencies ==============================
t0 = time.time()
run([sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off",
     "open_clip_torch", "wilds", "wandb", "braceexpand", "webdataset", "h5py", "huggingface_hub",
     "git+https://github.com/modestyachts/ImageNetV2_pytorch"], log_path=RESULTS / "pip.log")
log(f"dependencies installed in {fmt_t(time.time() - t0)}")

import importlib.metadata as md_
import torch
env = {
    "time": str(datetime.now()), "python": platform.python_version(), "platform": platform.platform(),
    "cpu_count": os.cpu_count(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "mem_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)}
             for i in range(torch.cuda.device_count())],
    "packages": {p: md_.version(p) for p in ["torchvision", "open_clip_torch", "wilds", "webdataset", "huggingface_hub", "pandas", "numpy"]},
    "scratch_free_gb": round(shutil.disk_usage(SCRATCH).free / 1e9, 1),
}
(RESULTS / "env.json").write_text(json.dumps(env, indent=2))
log("environment: " + json.dumps(env))
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU in the Kaggle notebook settings."
if env["scratch_free_gb"] < 35:
    log(f"WARNING: only {env['scratch_free_gb']} GB free in scratch; 35 GB recommended (archive + extracted data + checkpoints)")

2026-10-21 12:49:09 | $ /usr/bin/python3 -m pip install -q --progress-bar off open_clip_torch wilds wandb braceexpand webdataset h5py huggingface_hub git+https://github.com/modestyachts/ImageNetV2_pytorch
2026-10-21 12:49:18 | dependencies installed in 0:00:08
2026-10-21 12:49:23 | environment: {"time": "2026-10-21 12:49:22.695627", "python": "3.13.15", "platform": "Linux-6.18.48+-x86_64-with-glibc2.39", "cpu_count": 4, "torch": "2.11.0+cu128", "cuda": "12.8", "cuda_available": true, "gpus": [{"name": "Tesla T4", "mem_gb": 15.6}, {"name": "Tesla T4", "mem_gb": 15.6}], "packages": {"torchvision": "0.26.0+cu128", "open_clip_torch": "3.3.0", "wilds": "2.0.0", "webdataset": "1.0.2", "huggingface_hub": "1.29.0", "pandas": "2.3.3", "numpy": "2.1.3"}, "scratch_free_gb": 1146.0}


In [4]:
# ============================== W&B connectivity check (fails fast) ==============================
# Credentials: Kaggle secret (Add-ons > Secrets) or env var WANDB_API_KEY. Team/project come from the config cell.
if WANDB_ENABLED:
    import wandb
    key = get_secret("WANDB_API_KEY")
    assert key, "WANDB_API_KEY not found: add it as a Kaggle secret (attach it to this notebook) or set the env var, or set WANDB_ENABLED = False"
    assert wandb.login(key=key, relogin=True, verify=True), "W&B login failed: check the API key"
    _r = wandb.init(entity=WANDB_ENTITY or None, project=WANDB_PROJECT, name=f"{RUN_NAME}-connectivity-check",
                    job_type="check", tags=["check"], reinit="finish_previous")
    _r.log({"check/ok": 1})
    log(f"W&B OK: entity={_r.entity} project={_r.project} -> {_r.url}")
    _r.finish()
else:
    log("W&B disabled")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nnoukastephen (team-support-vectors-idlf26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261021_124925-b68302ty
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run iwildcam_flyp_vitb16-connectivity-check
wandb: ⭐️ View project at https://wandb.ai/team-support-vectors-idlf26/iwilds
wandb: 🚀 View run at https://wandb.ai/team-support-vectors-idlf26/iwilds/runs/b68302ty


2026-10-21 12:49:27 | W&B OK: entity=team-support-vectors-idlf26 project=iwilds -> https://wandb.ai/team-support-vectors-idlf26/iwilds/runs/b68302ty


wandb: updating run metadata; uploading summary
wandb: uploading wandb-summary.json
wandb: 
wandb: Run history:
wandb: check/ok ▁
wandb: 
wandb: Run summary:
wandb: check/ok 1
wandb: 
wandb: 🚀 View run iwildcam_flyp_vitb16-connectivity-check at: https://wandb.ai/team-support-vectors-idlf26/iwilds/runs/b68302ty
wandb: ⭐️ View project at: https://wandb.ai/team-support-vectors-idlf26/iwilds
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261021_124925-b68302ty/logs


In [5]:
# ============ clone FLYP at the pinned commit and patch it for progress logging ============
if not (REPO / ".git").exists():
    run(["git", "clone", "-q", REPO_URL, REPO])
run(["git", "checkout", "-q", REPO_COMMIT], cwd=REPO)
run(["git", "checkout", "-q", "--", "."], cwd=REPO)          # start from a clean tree so patches apply once

def patch(rel, old, new):
    path = REPO / rel
    text = path.read_text()
    assert text.count(old) == 1, f"patch target not found exactly once in {rel}: {old!r}"
    path.write_text(text.replace(old, new))

# Behaviour is unchanged unless the FLYP_* env vars below are set (the smoke test sets them).
patch("src/models/flyp_loss.py", "    print_every = 100\n",
      '    print_every = int(os.environ.get("FLYP_PRINT_EVERY", 100))\n')
patch("src/models/flyp_loss.py", "num_batches = len(dataset.train_loader)",
      'num_batches = min(len(dataset.train_loader), int(os.environ.get("FLYP_MAX_STEPS", 10**9)))')
patch("src/models/flyp_loss.py", 'f"ID FLYP Loss: {ft_clip_loss.item():.4f}")',
      'f"ID FLYP Loss: {ft_clip_loss.item():.4f}\\tbatch_time: {time.time() - start_time:.2f}s")')
patch("src/models/eval.py", "import numpy as np\nfrom src.models import utils",
      "import numpy as np\nfrom tqdm import tqdm\nfrom src.models import utils")
patch("src/models/eval.py", "    batched_data = enumerate(dataloader)\n",
      '    batched_data = enumerate(tqdm(dataloader, desc=f"eval {type(dataset).__name__}"))\n')
patch("src/models/eval.py", "        for i, data in batched_data:\n",
      "        for i, data in batched_data:\n"
      '            if i >= int(os.environ.get("FLYP_MAX_EVAL_BATCHES", 10**9)):\n'
      "                break\n")
patch("src/datasets/iwildcam.py", "num_workers=16,",
      'num_workers=int(os.environ.get("FLYP_NUM_WORKERS", 16)),')

# Seed everything from --seed (the FLYP code parses --seed but never uses it).
patch("src/main.py", "def main(args):\n",
      "def main(args):\n    random.seed(args.seed); np.random.seed(args.seed)\n"
      "    torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)\n")

# Mixed precision (FLYP_AMP=1). autocast must live inside the model's forward because DataParallel runs replicas in
# worker threads; outputs are cast back to fp32 so the loss and the fp32 classification head are unchanged.
patch("src/models/modeling.py", "import copy\n", "import copy\nimport os\n")
patch("src/models/modeling.py", "        return self.model(images, text)\n",
      '        with torch.autocast("cuda", dtype=torch.float16, enabled=os.environ.get("FLYP_AMP") == "1"):\n'
      "            out = self.model(images, text)\n"
      "        return tuple(o.float() for o in out) if isinstance(out, tuple) else out.float()\n")
patch("src/models/flyp_loss.py", "    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=args.wd)\n",
      "    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=args.wd)\n"
      '    scaler = torch.amp.GradScaler("cuda", enabled=os.environ.get("FLYP_AMP") == "1")\n')
patch("src/models/flyp_loss.py", "            ft_clip_loss.backward()\n            optimizer.step()\n",
      "            scaler.scale(ft_clip_loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n")

# Launcher: same as `python src/main.py`, but the training logger also streams to stdout.
(REPO / "run_flyp.py").write_text(
    "import logging, os, sys\n"
    "logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', handlers=[logging.StreamHandler(sys.stdout)])\n"
    "sys.path.insert(0, os.getcwd())\n"
    "from src.args import parse_arguments\n"
    "from src.main import main\n"
    "main(parse_arguments())\n")

# Prediction dump for the WILDS leaderboard: one csv of predicted class ids per split, in dataset order.
(REPO / "predict_flyp.py").write_text('''import os, sys
sys.path.insert(0, os.getcwd())
from pathlib import Path
import numpy as np, torch
from tqdm import tqdm
from src.args import parse_arguments
from src.datasets.iwildcam import IWildCam
from src.models.modeling import CLIPEncoder
from src.models.utils import get_logits
from src.models.zeroshot import get_zeroshot_classifier

args = parse_arguments()
out = Path(os.environ["FLYP_PRED_DIR"]); out.mkdir(parents=True, exist_ok=True)
model = CLIPEncoder.load(args.clip_load).cuda().eval()
head = get_zeroshot_classifier(args, model.model).cuda().eval()   # same head construction as the per-epoch eval
model = torch.nn.DataParallel(model)
for split in ["val", "test", "id_val", "id_test"]:
    ds = IWildCam(model.module.val_preprocess, location=args.data_location, batch_size=args.batch_size, subset=split)
    preds = []
    with torch.no_grad():
        for batch in tqdm(ds.test_loader, desc=f"predict {split}"):
            preds.append(get_logits(batch[0].cuda(), model, head).argmax(dim=1).cpu())
    preds = torch.cat(preds).numpy()
    assert len(preds) == len(ds.test_dataset), (split, len(preds), len(ds.test_dataset))
    np.savetxt(out / f"iwildcam_split:{split}_seed:{args.seed}_epoch:last_pred.csv", preds, fmt="%d")
    print(f"wrote {split}: {len(preds)} predictions", flush=True)
''')

diff = subprocess.run(["git", "diff"], cwd=REPO, capture_output=True, text=True).stdout
(RESULTS / "repo_patches.diff").write_text(diff)
log(f"FLYP @ {REPO_COMMIT[:8]} patched ({diff.count(chr(10) + '+') } added lines); diff saved to results/repo_patches.diff")

2026-10-21 12:49:27 | $ git clone -q https://github.com/locuslab/FLYP.git /kaggle/working/FLYP
2026-10-21 12:49:28 | $ git checkout -q 215d5bb6feeda6675f60e5818abcb4f6465c83af
2026-10-21 12:49:28 | $ git checkout -q -- .
2026-10-21 12:49:29 | FLYP @ 215d5bb6 patched (23 added lines); diff saved to results/repo_patches.diff


In [6]:
# ============================ download the dataset from the HF Hub ============================
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

MARKER = DATA_DIR / ".extract_complete"
ARCHIVE = None

def sha256_file(path, chunk=1 << 24):
    h, done, size, t0, last = hashlib.sha256(), 0, Path(path).stat().st_size, time.time(), 0
    with open(path, "rb") as f, tqdm(total=size, unit="B", unit_scale=True, desc="sha256", mininterval=10) as bar:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b); bar.update(len(b))
    return h.hexdigest()

if MARKER.exists():
    log(f"dataset already extracted at {DATA_DIR}; skipping download")
else:
    t0 = time.time()
    log(f"downloading {HF_DATASET_REPO}/{HF_ARCHIVE} (~{ARCHIVE_SIZE/1e9:.1f} GB) to {SCRATCH / 'hf_download'}")
    ARCHIVE = hf_hub_download(HF_DATASET_REPO, HF_ARCHIVE, repo_type="dataset", local_dir=SCRATCH / "hf_download")
    got = Path(ARCHIVE).stat().st_size
    log(f"download done in {fmt_t(time.time() - t0)}: {got/1e9:.2f} GB")
    assert got == ARCHIVE_SIZE, f"size mismatch: {got} != {ARCHIVE_SIZE}"
    t0 = time.time()
    digest = sha256_file(ARCHIVE)
    assert digest == ARCHIVE_SHA256, f"sha256 mismatch: {digest}"
    log(f"sha256 verified in {fmt_t(time.time() - t0)}: {digest}")

2026-10-21 12:49:29 | downloading emmilly/iwildcam-wilds-v2/iwildcam_v2.0.tar.gz (~12.0 GB) to /kaggle/temp/hf_download


iwildcam_v2.0.tar.gz: reconstructing file:   0%|          |  0.00B / 12.0GB            

iwildcam_v2.0.tar.gz: downloading bytes:           |  0.00B            

2026-10-21 12:50:27 | download done in 0:00:57: 11.96 GB


sha256:   0%|          | 0.00/12.0G [00:00<?, ?B/s]

2026-10-21 12:51:13 | sha256 verified in 0:00:46: 05c872f46eee44315f79bc90f4098c69ae3799a2b95116db02292ea1b4e370d2


In [7]:
# ====================== extract (with progress) and verify the WILDS splits ======================
EXPECTED_SPLITS = {"train": 129809, "val": 14961, "test": 42791, "id_val": 7314, "id_test": 8154}

if not MARKER.exists():
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    DATA_DIR.mkdir(parents=True)
    t0, last_report, n_files = time.time(), 0.0, 0
    size = Path(ARCHIVE).stat().st_size
    with open(ARCHIVE, "rb") as raw, tarfile.open(fileobj=raw, mode="r|gz") as tar:   # streaming: one pass
        for member in tar:
            if hasattr(tarfile, "data_filter"):
                tar.extract(member, DATA_DIR, filter="data")
            else:
                tar.extract(member, DATA_DIR)
            n_files += 1
            now = time.time()
            if now - last_report > 15:
                frac = raw.tell() / size
                eta = (now - t0) * (1 - frac) / max(frac, 1e-6)
                log(f"extract {100*frac:5.1f}% | {n_files:,} entries | elapsed {fmt_t(now - t0)} | ETA {fmt_t(eta)}")
                last_report = now
    (DATA_DIR / "RELEASE_v2.0.txt").touch()          # tells wilds the data is already present
    log(f"extracted {n_files:,} entries in {fmt_t(time.time() - t0)}")
    MARKER.touch()
    os.remove(ARCHIVE)                               # free 12 GB of scratch
    shutil.rmtree(SCRATCH / "hf_download", ignore_errors=True)
    log("archive deleted to free scratch space")

# --- verify through the same code path FLYP uses (wilds.get_dataset) ---
(DATA_DIR / "RELEASE_v2.0.txt").touch()
from wilds import get_dataset
ds = get_dataset("iwildcam", root_dir=str(DATA_ROOT), download=False)
sizes = {s: len(ds.get_subset(s)) for s in EXPECTED_SPLITS}
log(f"WILDS split sizes: {sizes}")
assert sizes == EXPECTED_SPLITS, f"unexpected split sizes, expected {EXPECTED_SPLITS}"

import random
meta = pd.read_csv(DATA_DIR / "metadata.csv")
from PIL import Image
for fn in random.Random(0).sample(list(meta["filename"]), 200):
    with Image.open(DATA_DIR / "train" / fn) as im:
        im.load()
log("200 random images decode fine; dataset ready")

2026-10-21 12:51:13 | extract   0.0% | 1 entries | elapsed 0:00:00 | ETA 0:11:01
2026-10-21 12:51:28 | extract  12.1% | 26,208 entries | elapsed 0:00:15 | ETA 0:01:49
2026-10-21 12:51:47 | extract  22.0% | 47,985 entries | elapsed 0:00:33 | ETA 0:01:57
2026-10-21 12:52:02 | extract  33.2% | 72,500 entries | elapsed 0:00:48 | ETA 0:01:36
2026-10-21 12:52:17 | extract  38.6% | 84,152 entries | elapsed 0:01:03 | ETA 0:01:40
2026-10-21 12:52:34 | extract  46.5% | 101,393 entries | elapsed 0:01:20 | ETA 0:01:32
2026-10-21 12:52:49 | extract  57.4% | 125,141 entries | elapsed 0:01:35 | ETA 0:01:10
2026-10-21 12:53:04 | extract  63.5% | 138,386 entries | elapsed 0:01:50 | ETA 0:01:03
2026-10-21 12:53:19 | extract  69.1% | 150,642 entries | elapsed 0:02:05 | ETA 0:00:56
2026-10-21 12:53:37 | extract  77.2% | 168,277 entries | elapsed 0:02:23 | ETA 0:00:42
2026-10-21 12:53:52 | extract  86.4% | 188,156 entries | elapsed 0:02:38 | ETA 0:00:24
2026-10-21 12:54:07 | extract  90.1% | 196,290 entrie

/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


2026-10-21 12:54:41 | WILDS split sizes: {'train': 129809, 'val': 14961, 'test': 42791, 'id_val': 7314, 'id_test': 8154}
2026-10-21 12:54:42 | 200 random images decode fine; dataset ready


In [8]:
# ============ build the FLYP training csv (2 prompts per image -> 259,618 rows) ============
csv_path = REPO / "datasets/csv/iwildcam_v2.0/train.csv"
run([sys.executable, "datacreation_scripts/iwildcam.py",
     "--metadata", DATA_DIR / "metadata.csv", "--data_dir", DATA_DIR / "train", "--save_file", csv_path],
    cwd=REPO, log_path=RESULTS / "csv_build.log")
df = pd.read_csv(csv_path, sep="\t")
assert len(df) == 129809 * 2
assert Path(df["filepath"].iloc[0]).exists()
log(f"training csv ready: {len(df):,} rows, {df['label'].nunique()} classes -> {csv_path}")
display(df.head(3))

2026-10-21 12:54:42 | $ /usr/bin/python3 datacreation_scripts/iwildcam.py --metadata /kaggle/temp/data/iwildcam_v2.0/metadata.csv --data_dir /kaggle/temp/data/iwildcam_v2.0/train --save_file /kaggle/working/FLYP/datasets/csv/iwildcam_v2.0/train.csv
2026-10-21 12:54:45 | training csv ready: 259,618 rows, 182 classes -> /kaggle/working/FLYP/datasets/csv/iwildcam_v2.0/train.csv


,title,filepath,label
0,a photo of empty.,/kaggle/temp/data/iwildcam_v2.0/train/87d70644...,0
1,a photo of empty.,/kaggle/temp/data/iwildcam_v2.0/train/975e8182...,0
2,a photo of empty.,/kaggle/temp/data/iwildcam_v2.0/train/934f9414...,0


In [9]:
# ============================== training + results persistence ==============================
import numpy as np
TRAIN_RE = re.compile(r"Train Epoch: (\d+) \[(\d+)% (\d+)/(\d+)\]")
LOSS_RE  = re.compile(r"ID FLYP Loss: ([\d.]+)(?:\s+batch_time: ([\d.]+)s)?")
SPLITS   = ["val", "test", "id_val", "id_test"]
PRED_DIR = RESULTS / "submission" / "iwildcam"

def pred_file(split, seed):
    return PRED_DIR / f"iwildcam_split:{split}_seed:{seed}_epoch:last_pred.csv"

def prune_checkpoints(exp, seed):
    d = CKPT / exp
    pat = re.compile(r"(?:checkpoint|optim)_(\d+)\.pt$")
    files = [(int(m.group(1)), f) for f in d.glob(f"*_run{seed}/*.pt") if (m := pat.search(f.name))]
    keep = sorted({e for e, _ in files})[-KEEP_LAST_CHECKPOINTS:]
    for e, f in files:
        if e not in keep:
            f.unlink()

def hf_upload():
    if not HF_LOG_REPO:
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=get_secret("HF_TOKEN"))
        api.create_repo(HF_LOG_REPO, repo_type="dataset", private=True, exist_ok=True)
        api.upload_folder(folder_path=str(RESULTS), repo_id=HF_LOG_REPO, repo_type="dataset",
                          path_in_repo=RUN_NAME, commit_message=f"{RUN_NAME} {datetime.now():%F %T}")
        log(f"logs pushed to hf.co/datasets/{HF_LOG_REPO}/{RUN_NAME}")
    except Exception as e:                         # never let a log upload kill training
        log(f"WARNING: HF log upload failed: {e!r}")

# ----------------------------- Weights & Biases (never allowed to kill training) -----------------------------
WB, WB_LOGGED = None, set()

def wandb_start(seed):
    global WB
    if not WANDB_ENABLED:
        return
    try:
        import wandb
        key = get_secret("WANDB_API_KEY")
        if not key:
            log("WARNING: WANDB_API_KEY not found (Kaggle secret or env var); W&B logging disabled")
            return
        wandb.login(key=key, relogin=True)
        WB = wandb.init(entity=WANDB_ENTITY or None, project=WANDB_PROJECT, group=RUN_NAME, name=f"{RUN_NAME}-seed{seed}",
                        job_type="train", tags=[PRESET, MODEL] + (["dryrun"] if DRY_RUN_STEPS else []),
                        config=dict(run_name=RUN_NAME, preset=PRESET, model=MODEL, epochs=EPOCHS, batch_size=BATCH_SIZE, amp=AMP,
                                    lr=LR, wd=WD, seed=seed, repo_commit=REPO_COMMIT, exp_name=EXP_NAME))
        wandb.define_metric("train/step")
        wandb.define_metric("train/*", step_metric="train/step")
        wandb.define_metric("epoch")
        wandb.define_metric("eval/*", step_metric="epoch")
        log(f"W&B run: {WB.url}")
    except Exception as e:
        WB = None
        log(f"WARNING: W&B init failed, continuing without it: {e!r}")

def wandb_log(data):
    if WB is None:
        return
    try:
        WB.log(data)
    except Exception as e:
        log(f"WARNING: W&B log failed: {e!r}")

def wandb_finish(seed, final=None):
    global WB
    if WB is None:
        return
    try:
        import wandb
        if final:
            WB.summary.update(final)
        art = wandb.Artifact(f"{RUN_NAME}-seed{seed}-logs", type="logs")
        for p in RESULTS.rglob("*"):
            if p.is_file() and "submission" not in p.parts and p.suffix in {".log", ".json", ".tsv", ".csv", ".diff"} \
                    and p.stat().st_size < 50e6:
                art.add_file(str(p), name=str(p.relative_to(RESULTS)))
        WB.log_artifact(art)
        WB.finish()
    except Exception as e:
        log(f"WARNING: W&B finish failed: {e!r}")
    WB = None

def sync_results(exp, seed=0):
    """Copy the repo's per-epoch logs into the saved output, report the latest epoch, trim checkpoints."""
    src, dst = REPO / "expt_logs" / exp, RESULTS / "expt_logs" / exp
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    for stats in sorted(dst.glob(f"*_run{seed}/stats.tsv")):
        df = pd.read_csv(stats, sep="\t")
        for _, r in df.iterrows():
            row = {k: v for k, v in r.to_dict().items() if not str(k).startswith("Unnamed")}
            if WB is not None and (seed, int(row["epoch"])) not in WB_LOGGED:
                WB_LOGGED.add((seed, int(row["epoch"])))
                wandb_log({"epoch": int(row["epoch"]), **{f"eval/{k}": float(v) for k, v in row.items() if k != "epoch"}})
        log(f"epoch finished: {json.dumps({k: v for k, v in df.iloc[-1].to_dict().items() if not str(k).startswith('Unnamed')})}")
    prune_checkpoints(exp, seed)
    hf_upload()

class TrainMonitor:
    """Adds '% of run / elapsed / ETA' to training lines, streams losses to W&B, syncs results after every epoch."""
    def __init__(self, exp, epochs, seed):
        self.exp, self.epochs, self.seed, self.t0, self.pending = exp, epochs, seed, time.time(), False
    def __call__(self, line):
        note = None
        if self.pending:                            # stats.tsv is written right after the 'Avg ID FLYP Loss' line
            self.pending = False
            sync_results(self.exp, self.seed)
        m = TRAIN_RE.search(line)
        if m:
            e, _, i, n = map(int, m.groups())
            frac = (e + i / n) / self.epochs
            el = time.time() - self.t0
            eta = f"{fmt_t(el * (1 - frac) / frac)}" if frac > 0.01 else "?"
            note = f"[run {100*frac:5.1f}% | elapsed {fmt_t(el)} | ETA~{eta}]"
            l = LOSS_RE.search(line)
            if l and WB is not None:
                d = {"train/step": e * n + i, "train/epoch": e + i / n, "train/flyp_loss": float(l.group(1))}
                if l.group(2):
                    d["train/batch_time_s"] = float(l.group(2))
                wandb_log(d)
        if "Avg ID FLYP Loss" in line:
            self.pending = True
        return note

def flyp_args(exp, epochs, seed):
    return ["--train-dataset=IWildCamIDVal", f"--epochs={epochs}", f"--lr={LR}", f"--wd={WD}",
            f"--batch-size={BATCH_SIZE}", f"--model={MODEL}", "--eval-datasets=IWildCamIDVal,IWildCamID,IWildCamOOD",
            "--template=iwildcam_template", f"--save={CKPT}/", f"--data-location={DATA_ROOT}/",
            f"--ft_data={csv_path}", "--csv-img-key=filepath", "--csv-caption-key=title",
            f"--exp_name={exp}", f"--workers={WORKERS}", f"--run={seed}", f"--seed={seed}"]

def train(exp, epochs, seed=0, extra_env=None):
    env = {"PYTHONPATH": str(REPO), "FLYP_PRINT_EVERY": str(PRINT_EVERY), "FLYP_NUM_WORKERS": str(WORKERS),
           "TQDM_MININTERVAL": "30", "PYTHONUNBUFFERED": "1", "FLYP_AMP": str(int(AMP)), **(extra_env or {})}
    mon = TrainMonitor(exp, epochs, seed)
    try:
        run([sys.executable, "-u", "run_flyp.py", *flyp_args(exp, epochs, seed)], cwd=REPO, env=env, hook=mon,
            log_path=RESULTS / f"train_stdout_{exp.split('/')[0]}_seed{seed}.log")
    finally:
        sync_results(exp, seed)                     # keep whatever finished, even if the run died

# ----------------------------- leaderboard predictions -----------------------------
def verify_predictions(seed):
    """Re-score the prediction csvs with the WILDS evaluator (same code the leaderboard uses)."""
    from wilds import get_dataset
    d = get_dataset("iwildcam", root_dir=str(DATA_ROOT), download=False)
    final = {}
    for split in SPLITS:
        p = torch.as_tensor(np.loadtxt(pred_file(split, seed), dtype=int, ndmin=1))
        sub = d.get_subset(split)
        assert len(p) == len(sub), f"{split}: {len(p)} predictions for {len(sub)} examples"
        res = d.eval(p, sub.y_array, sub.metadata_array)[0]
        log(f"[verify seed {seed}] {split}: acc {res['acc_avg']:.4f} | macro-F1 {res['F1-macro_all']:.4f}")
        final[f"final/{split}_acc"], final[f"final/{split}_macro_f1"] = res["acc_avg"], res["F1-macro_all"]
    return final

def predict(seed):
    pat = re.compile(r"checkpoint_(\d+)\.pt$")
    ckpts = sorted((CKPT / EXP_NAME).glob(f"*_run{seed}/checkpoint_*.pt"), key=lambda p: int(pat.search(p.name).group(1)))
    assert ckpts and int(pat.search(ckpts[-1].name).group(1)) == EPOCHS - 1, "final-epoch checkpoint missing"
    log(f"predicting {SPLITS} for seed {seed} with {ckpts[-1]}")
    run([sys.executable, "-u", "predict_flyp.py", *flyp_args(EXP_NAME, EPOCHS, seed), f"--clip_load={ckpts[-1]}"], cwd=REPO,
        env={"PYTHONPATH": str(REPO), "FLYP_NUM_WORKERS": str(WORKERS), "FLYP_PRED_DIR": str(PRED_DIR),
             "TQDM_MININTERVAL": "30", "PYTHONUNBUFFERED": "1", "FLYP_AMP": str(int(AMP))},
        log_path=RESULTS / f"predict_seed{seed}.log")
    return verify_predictions(seed)

def package_submission():
    """Bundle predictions/iwildcam/*.csv into a .tar.gz for the WILDS submission form."""
    PRED_DIR.mkdir(parents=True, exist_ok=True)
    for d in EXTRA_PRED_DIRS:
        for f in Path(d).glob("*_pred.csv"):
            shutil.copy(f, PRED_DIR / f.name)
    seeds = sorted({int(re.search(r"seed:(\d+)", f.name).group(1)) for f in PRED_DIR.glob("*_pred.csv")})
    missing = [pred_file(sp, s).name for s in seeds for sp in SPLITS if not pred_file(sp, s).exists()]
    assert seeds and not missing, f"incomplete predictions: {missing}"
    if len(seeds) < 3:
        log(f"WARNING: only seeds {seeds}; the leaderboard requires at least 3 seeds. Not submittable yet.")
    out = RESULTS / "wilds_submission.tar.gz"
    with tarfile.open(out, "w:gz") as tar:
        tar.add(PRED_DIR, arcname="predictions/iwildcam")
    log(f"submission archive: {out} ({out.stat().st_size/1e6:.1f} MB, seeds {seeds}). Upload it (Drive/Dropbox) and fill the form at https://wilds.stanford.edu/submit/")

(RESULTS / "config.json").write_text(json.dumps(
    dict(RUN_NAME=RUN_NAME, EXP_NAME=EXP_NAME, REPO_COMMIT=REPO_COMMIT, PRESET=PRESET, MODEL=MODEL, EPOCHS=EPOCHS,
         BATCH_SIZE=BATCH_SIZE, AMP=AMP, LR=LR, WD=WD, SEEDS=SEEDS, WORKERS=WORKERS, HF_DATASET_REPO=HF_DATASET_REPO,
         ARCHIVE_SHA256=ARCHIVE_SHA256, WANDB_ENTITY=WANDB_ENTITY, WANDB_PROJECT=WANDB_PROJECT), indent=2))
log("training helpers ready")


2026-10-21 12:54:45 | training helpers ready


In [10]:
# =============== smoke test: a few steps at the real batch size (catches OOM / path errors) ===============
if RUN_SMOKE_TEST_FIRST:
    log(f"SMOKE TEST: {SMOKE_STEPS} train steps, {SMOKE_EVAL_BATCHES} eval batches per split, batch size {BATCH_SIZE}")
    train("smoke/flyp_loss", 1, extra_env={"FLYP_MAX_STEPS": str(SMOKE_STEPS), "FLYP_MAX_EVAL_BATCHES": str(SMOKE_EVAL_BATCHES),
                                  "FLYP_PRINT_EVERY": "1"})
    smoke = list((REPO / "expt_logs/smoke/flyp_loss").glob("*/stats.tsv"))
    assert smoke and "IWildCamOOD Accuracy" in pd.read_csv(smoke[0], sep="\t").columns, "smoke test produced no OOD metrics"
    shutil.rmtree(CKPT / "smoke", ignore_errors=True)
    log("SMOKE TEST PASSED (its numbers are meaningless; real results are logged under iwildcam/)")
else:
    log("smoke test skipped")

2026-10-21 12:54:45 | SMOKE TEST: 3 train steps, 3 eval batches per split, batch size 64
2026-10-21 12:54:45 | $ /usr/bin/python3 -u run_flyp.py --train-dataset=IWildCamIDVal --epochs=1 --lr=1e-05 --wd=0.2 --batch-size=64 --model=ViT-B/16 --eval-datasets=IWildCamIDVal,IWildCamID,IWildCamOOD --template=iwildcam_template --save=/kaggle/temp/checkpoints/ --data-location=/kaggle/temp/data/ --ft_data=/kaggle/working/FLYP/datasets/csv/iwildcam_v2.0/train.csv --csv-img-key=filepath --csv-caption-key=title --exp_name=smoke/flyp_loss --workers=4 --run=0 --seed=0
/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
2026-10-21 12:54:50,098 NumExpr defaulting to 4 threads.
ViT-B/16
  0%|                    

## Full training run

Progress lines look like `Train Epoch: 3 [40% 200/507] ... [run 17.8% | elapsed 1:20:11 | ETA~6:10:00]`. After each epoch the metrics row is logged
(`epoch finished: {...}`) and copied to `/kaggle/working/results/...`, so a crash or timeout keeps everything finished so far.
An earlier fp32, batch-64 run measured ~1 h/epoch on 2x T4; AMP at batch 256 is expected to be faster but is unmeasured. 20 epochs may not fit a 12 h session: check the first ETA and lower `EPOCHS` if needed (the cosine schedule follows it).

In [11]:
log(f"FULL RUN: {PRESET} | {EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}, wd {WD}, {MODEL}, seeds {SEEDS}")
for seed in SEEDS:
    log(f"=== seed {seed} ===")
    wandb_start(seed)
    final = None
    try:
        train(EXP_NAME, EPOCHS, seed, extra_env={"FLYP_MAX_STEPS": str(DRY_RUN_STEPS)} if DRY_RUN_STEPS else None)
        final = predict(seed)
    finally:
        wandb_finish(seed, final)
log("training finished")
package_submission()


2026-10-21 12:55:55 | FULL RUN: baseline | 20 epochs, batch 64, lr 1e-05, wd 0.2, ViT-B/16, seeds [0]
2026-10-21 12:55:55 | === seed 0 ===


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261021_125556-r1pvlmht
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run iwildcam_flyp_vitb16-seed0
wandb: ⭐️ View project at https://wandb.ai/team-support-vectors-idlf26/iwilds
wandb: 🚀 View run at https://wandb.ai/team-support-vectors-idlf26/iwilds/runs/r1pvlmht


2026-10-21 12:55:57 | W&B run: https://wandb.ai/team-support-vectors-idlf26/iwilds/runs/r1pvlmht
2026-10-21 12:55:57 | $ /usr/bin/python3 -u run_flyp.py --train-dataset=IWildCamIDVal --epochs=20 --lr=1e-05 --wd=0.2 --batch-size=64 --model=ViT-B/16 --eval-datasets=IWildCamIDVal,IWildCamID,IWildCamOOD --template=iwildcam_template --save=/kaggle/temp/checkpoints/ --data-location=/kaggle/temp/data/ --ft_data=/kaggle/working/FLYP/datasets/csv/iwildcam_v2.0/train.csv --csv-img-key=filepath --csv-caption-key=title --exp_name=iwildcam/flyp_loss --workers=4 --run=0 --seed=0
/usr/local/lib/python3.13/dist-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version
2026-10-21 12:56:00,823 NumExpr defaulting to 4 threads.
ViT-B/16
2026-10-21 12

wandb: updating run metadata; uploading artifact iwildcam_flyp_vitb16-seed0-logs
wandb: uploading artifact iwildcam_flyp_vitb16-seed0-logs
wandb: uploading summary, console lines 2486-2492
wandb: 
wandb: Run history:
wandb:                          epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:          eval/Avg ID FLYP Loss █▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
wandb:               eval/Avg OOD Acc ▄▆▆▁▆▇▆▅▇▇█▇▇███████
wandb:       eval/IWildCamID Accuracy ▁▄▄▁▅▆▆▄▆▆▇▆████████
wandb:    eval/IWildCamIDF1-macro_all ▁▁▁▂▃▃▄▃▅▅▇▆▇▆▇▇▇▇██
wandb:    eval/IWildCamIDVal Accuracy ▄▆▆▁▇▇▇▅▇▇█▇▇███████
wandb: eval/IWildCamIDValF1-macro_all ▂▃▁▂▃▄▃▂▄▆▇▆▆▇▇██▇██
wandb:      eval/IWildCamOOD Accuracy ▅▆▇▁▆▆▆▅▆▆█▆▇▇▇▇▇▇▇▇
wandb:   eval/IWildCamOODF1-macro_all ▂▂▄▂▁▃▃▂▄▅█▆▆▆▇█▇▇██
wandb:             train/batch_time_s ▃▃▁▆▁▆▆▆▃▃▆▆▁▆▃▆▆▃▃▆▆▃▃▆▃▃▆█▁▆▃▃▆▃▃▆▃▃▁▃
wandb:                             +3 ...
wandb: 
wandb: Run summary:
wandb:                          epoch 19
wandb:          eval/Avg ID FLYP Loss 1.6778
wandb:           

2026-10-21 20:48:06 | training finished
2026-10-21 20:48:06 | WARNING: only seeds [0]; the leaderboard requires at least 3 seeds. Not submittable yet.
2026-10-21 20:48:06 | submission archive: /kaggle/working/results/iwildcam_flyp_vitb16/wilds_submission.tar.gz (0.0 MB, seeds [0]). Upload it (Drive/Dropbox) and fill the form at https://wilds.stanford.edu/submit/


In [12]:
# ================================== final summary ==================================
runs = {}
for sf in sorted((RESULTS / "expt_logs" / EXP_NAME).glob("*/stats.tsv")):
    runs[int(re.search(r"_run(\d+)$", sf.parent.name).group(1))] = pd.read_csv(sf, sep="\t", index_col=0)
assert runs, "no stats.tsv found, training did not complete an epoch"
pd.set_option("display.width", 250, "display.max_columns", 50)
for seed, stats in runs.items():
    print(f"seed {seed}")
    display(stats)
    stats.to_csv(RESULTS / f"per_epoch_metrics_seed{seed}.csv")

finals = pd.DataFrame({seed: s.iloc[-1] for seed, s in runs.items()}).T
if len(finals) > 1:
    display(finals.agg(["mean", "std"]))
summary = {"run": RUN_NAME, "epochs_planned": EPOCHS, "final_epoch_metrics_by_seed": finals.to_dict(orient="index"),
           "config": json.loads((RESULTS / "config.json").read_text())}
(RESULTS / "results_summary.json").write_text(json.dumps(summary, indent=2))
log("FINAL (last epoch): " + json.dumps(finals.to_dict(orient="index")))
for seed in runs:
    sync_results(EXP_NAME, seed)
if SAVE_FINAL_CHECKPOINT_TO_OUTPUT:
    last = sorted((CKPT / EXP_NAME).glob("**/checkpoint_*.pt"), key=lambda p: int(re.search(r"_(\d+)\.pt", p.name).group(1)))[-1]
    shutil.copy(last, RESULTS / last.name)
    log(f"copied {last.name} to results")
log(f"all results are in {RESULTS}")


seed 0


,epoch,IWildCamIDVal Accuracy,IWildCamIDValF1-macro_all,IWildCamID Accuracy,IWildCamIDF1-macro_all,IWildCamOOD Accuracy,IWildCamOODF1-macro_all,Avg OOD Acc,Avg ID FLYP Loss
0,0,0.5269,0.3281,0.4709,0.3740,0.5318,0.2494,0.5099,2.1153
1,1,0.6466,0.3388,0.5579,0.3669,0.5898,0.2502,0.5981,1.8502
2,2,0.6254,0.3032,0.5636,0.3712,0.6300,0.2697,0.6063,1.7815
3,3,0.3671,0.3301,0.4671,0.3898,0.3427,0.2498,0.3923,1.7630
4,4,0.6891,0.3463,0.6148,0.4103,0.5782,0.2437,0.6274,1.7376
5,5,0.7142,0.3548,0.6545,0.4011,0.5941,0.2652,0.6543,1.7294
6,6,0.6687,0.3415,0.6285,0.4232,0.5662,0.2645,0.6211,1.7201
7,7,0.6024,0.3226,0.5660,0.4080,0.5301,0.2503,0.5662,1.7105
8,8,0.6777,0.3639,0.6532,0.4309,0.5771,0.2695,0.6360,1.7036
9,9,0.6799,0.3915,0.6463,0.4305,0.5906,0.2820,0.6389,1.6998


2026-10-21 20:48:06 | FINAL (last epoch): {"0": {"epoch": 19.0, "IWildCamIDVal Accuracy": 0.7428, "IWildCamIDValF1-macro_all": 0.4317, "IWildCamID Accuracy": 0.7107, "IWildCamIDF1-macro_all": 0.4927, "IWildCamOOD Accuracy": 0.6336, "IWildCamOODF1-macro_all": 0.3084, "Avg OOD Acc": 0.6957, "Avg ID FLYP Loss": 1.6778}}
2026-10-21 20:48:06 | epoch finished: {"epoch": 19.0, "IWildCamIDVal Accuracy": 0.7428, "IWildCamIDValF1-macro_all": 0.4317, "IWildCamID Accuracy": 0.7107, "IWildCamIDF1-macro_all": 0.4927, "IWildCamOOD Accuracy": 0.6336, "IWildCamOODF1-macro_all": 0.3084, "Avg OOD Acc": 0.6957, "Avg ID FLYP Loss": 1.6778}
2026-10-21 20:48:06 | all results are in /kaggle/working/results/iwildcam_flyp_vitb16
